# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)
# Resampling techniques (pip install imbalanced-learn if it is not installed)
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [3]:
# Step 1: distribution of the target variable
counts = fraud["fraud"].value_counts()
percent = fraud["fraud"].value_counts(normalize=True).mul(100).round(2)

print(pd.DataFrame({"transactions": counts, "percentage_%": percent}))
print(f"\nImbalance ratio (legit : fraud) = {counts.max() / counts.min():.1f} : 1")

       transactions  percentage_%
fraud                            
0.0          912597         91.26
1.0           87403          8.74

Imbalance ratio (legit : fraud) = 10.4 : 1


1. The database has 91.26% legit(0) vs 8.74% fraud(1). That's roughly a 10.4:1 ratio - yes, this is a clearly imbalanced dataset. With this imbalance, accuracy alone is misleading.

In [4]:
# Step 2: split (stratified, so both sets keep the same fraud percentage) and scale
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on the training data only
X_test_s = scaler.transform(X_test)         # the test set is never resampled

print("Train distribution:", y_train.value_counts().to_dict())
print("Test distribution :", y_test.value_counts().to_dict())

Train distribution: {0.0: 730078, 1.0: 69922}
Test distribution : {0.0: 182519, 1.0: 17481}


In [5]:
results = {}   # metrics of every model, to compare them at the end

def evaluate(model, X_te, y_te, label):
    """Evaluate a fitted classifier with imbalance-aware metrics and return them."""
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()

    print(f"===== {label} =====")
    print(classification_report(y_te, y_pred, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_te, y_pred))
    print(f"Precision (fraud=1): {precision_score(y_te, y_pred):.4f}")
    print(f"Recall (fraud=1): {recall_score(y_te, y_pred):.4f}")
    print(f"F1 (fraud=1): {f1_score(y_te, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_te, y_proba):.4f}")
    print(f"PR-AUC (avg prec): {average_precision_score(y_te, y_proba):.4f}")

    return {
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "precision": precision_score(y_te, y_pred),
        "recall": recall_score(y_te, y_pred),
        "f1": f1_score(y_te, y_pred),
        "roc_auc": roc_auc_score(y_te, y_proba),
        "pr_auc": average_precision_score(y_te, y_proba),
        "missed_frauds": fn,
        "false_alarms": fp,
    }

In [6]:
# Step 2 / 3: baseline model on the imbalanced data
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_s, y_train)
results["Baseline"] = evaluate(log_reg, X_test_s, y_test, "BASELINE (imbalanced)")

===== BASELINE (imbalanced) =====
              precision    recall  f1-score   support

         0.0     0.9634    0.9933    0.9781    182519
         1.0     0.8964    0.6056    0.7228     17481

    accuracy                         0.9594    200000
   macro avg     0.9299    0.7994    0.8505    200000
weighted avg     0.9575    0.9594    0.9558    200000

Confusion matrix:
 [[181296   1223]
 [  6895  10586]]
Precision (fraud=1): 0.8964
Recall (fraud=1): 0.6056
F1 (fraud=1): 0.7228


ROC-AUC: 0.9670
PR-AUC (avg prec): 0.8072


**3. Evaluation of the baseline.** Accuracy (95.94%) looks great but is misleading: a model that always answers "legit" would already score 91.26%. What matters is the fraud class. The model misses 6,895 of the 17,481 frauds in the test set (recall 60.56%), while being right 89.64% of the time when it flags a fraud (1,223 false alarms). F1 for fraud is 72.28%.

Because missing a fraud is usually much more costly than a false alarm, the key metric is **recall of the fraud class**, read together with precision, F1 and PR-AUC to see the price paid for it. ROC-AUC (0.967) is high but too optimistic on imbalanced data, and accuracy should not be used.

In [7]:
# Step 4: oversampling (only the TRAINING data is resampled; the test set stays untouched)
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_s, y_train)
print("\nOversampled train distribution:\n", y_train_ros.value_counts())
log_reg_ros = LogisticRegression(max_iter=1000, random_state=42)
log_reg_ros.fit(X_train_ros, y_train_ros)
results["Oversampling"] = evaluate(log_reg_ros, X_test_s, y_test, "OVERSAMPLED (RandomOverSampler)")


Oversampled train distribution:
 fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


===== OVERSAMPLED (RandomOverSampler) =====
              precision    recall  f1-score   support

         0.0     0.9947    0.9335    0.9631    182519
         1.0     0.5774    0.9479    0.7176     17481

    accuracy                         0.9348    200000
   macro avg     0.7860    0.9407    0.8404    200000
weighted avg     0.9582    0.9348    0.9417    200000

Confusion matrix:
 [[170390  12129]
 [   911  16570]]
Precision (fraud=1): 0.5774
Recall (fraud=1): 0.9479
F1 (fraud=1): 0.7176
ROC-AUC: 0.9795


PR-AUC (avg prec): 0.7574


**4. Oversampling: does it improve the model?** Yes for detecting fraud, but with a trade-off. The training set now has 730,078 legit and 730,078 fraud rows. Recall of the fraud class jumps from 60.56% to 94.79% (missed frauds go from 6,895 to 911), but precision falls from 89.64% to 57.74% (false alarms go from 1,223 to 12,129) and accuracy drops from 95.94% to 93.48%. F1 stays about the same (72.28% vs 71.76%), ROC-AUC improves slightly (0.967 to 0.980) and PR-AUC drops (0.807 to 0.757). It is a better model if catching fraud matters more than avoiding false alarms.

In [8]:
# Step 5: undersampling
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_s, y_train)
print("\nUndersampled train distribution:\n", y_train_rus.value_counts())
log_reg_rus = LogisticRegression(max_iter=1000, random_state=42)
log_reg_rus.fit(X_train_rus, y_train_rus)
results["Undersampling"] = evaluate(log_reg_rus, X_test_s, y_test, "UNDERSAMPLED (RandomUnderSampler)")


Undersampled train distribution:
 fraud
0.0    69922
1.0    69922
Name: count, dtype: int64
===== UNDERSAMPLED (RandomUnderSampler) =====
              precision    recall  f1-score   support

         0.0     0.9946    0.9336    0.9631    182519
         1.0     0.5773    0.9475    0.7175     17481

    accuracy                         0.9348    200000
   macro avg     0.7860    0.9405    0.8403    200000
weighted avg     0.9582    0.9348    0.9417    200000



Confusion matrix:
 [[170394  12125]
 [   918  16563]]
Precision (fraud=1): 0.5773
Recall (fraud=1): 0.9475
F1 (fraud=1): 0.7175
ROC-AUC: 0.9796
PR-AUC (avg prec): 0.7569


**5. Undersampling: does it improve the model?** The result is practically the same as with oversampling: recall 94.75%, precision 57.73%, F1 71.75%, with 918 missed frauds and 12,125 false alarms. It brings the same improvement in recall and the same loss of precision compared with the baseline. The difference is that it trains on 69,922 legit and 69,922 fraud rows, about a tenth of the rows used by oversampling, since most legit transactions are thrown away; here that costs nothing in performance.

In [9]:
# Step 6: SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)
print("\nSMOTE train distribution:\n", y_train_sm.value_counts())
log_reg_sm = LogisticRegression(max_iter=1000, random_state=42)
log_reg_sm.fit(X_train_sm, y_train_sm)
results["SMOTE"] = evaluate(log_reg_sm, X_test_s, y_test, "SMOTE")   # same scaled test set as before


SMOTE train distribution:
 fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


===== SMOTE =====
              precision    recall  f1-score   support

         0.0     0.9947    0.9335    0.9631    182519
         1.0     0.5774    0.9481    0.7177     17481

    accuracy                         0.9348    200000
   macro avg     0.7860    0.9408    0.8404    200000
weighted avg     0.9582    0.9348    0.9417    200000

Confusion matrix:
 [[170386  12133]
 [   907  16574]]
Precision (fraud=1): 0.5774
Recall (fraud=1): 0.9481
F1 (fraud=1): 0.7177
ROC-AUC: 0.9795


PR-AUC (avg prec): 0.7574


**6. SMOTE: does it improve the model?** Again the result is virtually identical: recall 94.81%, precision 57.74%, F1 71.77%. Compared with the baseline it improves recall in the same way (and pays the same price in precision), but it gives no advantage over simple random oversampling or undersampling on this dataset, so its extra complexity is not justified here.

In [10]:
# Comparison of the four models on the same, untouched test set
comparison = pd.DataFrame(results).T
comparison[["missed_frauds", "false_alarms"]] = comparison[["missed_frauds", "false_alarms"]].astype(int)
comparison.round(4)

,accuracy,precision,recall,f1,roc_auc,pr_auc,missed_frauds,false_alarms
Baseline,0.9594,0.8964,0.6056,0.7228,0.9670,0.8072,6895,1223
Oversampling,0.9348,0.5774,0.9479,0.7176,0.9795,0.7574,911,12129
Undersampling,0.9348,0.5773,0.9475,0.7175,0.9796,0.7569,918,12125
SMOTE,0.9348,0.5774,0.9481,0.7177,0.9795,0.7574,907,12133


**Overall conclusion.** The dataset is imbalanced (91.26% legit vs 8.74% fraud), and the baseline logistic regression reaches a high accuracy while missing about 4 out of 10 frauds. Balancing the training data with any of the three techniques raises the fraud recall from about 61% to about 95%, at the cost of precision (about 90% down to about 58%) and more false alarms, while F1 does not improve. The three techniques perform the same here, so the simplest one (random under- or oversampling) is enough. Which model to prefer depends on the relative cost of a missed fraud versus a false alarm: if fraud is the bigger cost, use a balanced model; if reviewing false alarms is expensive, the baseline is the more precise option.